# Decoder comparison - OOF decoder comparison and evidence report

This notebook aggregates the twenty completed decoder runs from Stage 2.3 and compares the four decoder arms (a 2x2 of {ReLU, PReLU} activation x {plain, residual} skip topology) using paired out-of-fold predictions, decomposing the residual and activation main effects.

The scientific claim is deliberately narrow: performance is conditional on this 71-knee/43-subject cohort, the fold-specific frozen front ends, the matched training protocol, and seed 42. It is not a claim that one original architecture is universally superior or that either model is clinically validated.

## Success criteria

1. Exactly 71 unique OOF knees are present per arm and every knee is paired across all four arms with the same test fold and shared-front-end hash.
2. Bilateral knees are averaged within subject before inference, producing exactly 43 paired subject records.
3. Co-primary endpoints are subject-level macro Dice and macro ASSD in millimetres; no co-primary value is missing or infinite.
4. Two-sided paired Wilcoxon tests use Pratt zero handling, with Holm correction across the co-primary main-effect family (two contrasts x two endpoints).
5. Ten-thousand fold-stratified subject bootstrap resamples provide paired-effect 95% confidence intervals for every contrast.
6. A factor (residual or activation) is preferred only if both Holm-corrected co-primary endpoints are significant in the same direction and the residual x activation interaction is not significant; otherwise the result is `INCONCLUSIVE`.
7. Decoder-family diagonal, residual x activation interaction, pathology interaction, bone-wise, healthy/fractured, topology, parameter, memory, resource, and fold-0 seed-variance results are reported as secondary or descriptive evidence.
8. Regen outputs, if supplied, contain deidentified study IDs only and are labelled illustrative; they cannot select the winning decoder.

## Research-gap interpretation

[Kasten et al. (2020)](https://arxiv.org/abs/2004.00871) and [Lin et al. (2026)](https://pubmed.ncbi.nlm.nih.gov/41840145/) establish direct biplanar knee-bone reconstruction, while [Shakya and Khanal (NeurIPS 2023)](https://papers.neurips.cc/paper_files/paper/2023/hash/412732f172bdd5ad0efde2fafa110700-Abstract-Datasets_and_Benchmarks.html) already benchmark complete biplanar reconstruction systems and explicitly call for disaggregated clinically relevant subgroup reporting. This study therefore addresses a narrower, locally evidenced gap: among the studies reviewed for this project, none isolates matched plain-versus-residual decoding and ReLU-versus-PReLU activation in a 2x2 factorial under a byte-identical pretrained and frozen biplanar knee representation while separately reporting healthy and fractured knees. This is a scoped literature-review claim, not a claim of exhaustive systematic-review novelty.

Dice is paired with ASSD because [Metrics Reloaded (Maier-Hein et al., 2024)](https://doi.org/10.1038/s41592-023-02151-z) recommends problem-aware metric selection, including boundary-sensitive evidence when boundary accuracy is part of the domain interest and explicit handling of empty predictions. Fractured results remain descriptive because only 13 fractured knees are available.

The primary analysis retains all five folds. A mandatory, pre-registered sensitivity analysis excludes `test_fold == 4`; it cannot independently select a preferred factor.

## Run-provenance compatibility

Runs produced by the revised 03b decoder cross-validation notebook are accepted only when their embedded qa.stage2_training_evidence matches the current capacity-summary and protocol-amendment hashes and their registered weighted-loss policy. Runs without that evidence remain legacy candidates and require exact independent acceptance in the combined gate; unaccepted pre-amendment runs are rejected.


In [1]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import torch

STAGE2_SCHEMA="foundation_stage2_v1"
BONES=["femur","tibia","patella","fibula"]

def find_project_root(start):
    for candidate in [Path(start).resolve(),*Path(start).resolve().parents]:
        if (candidate/"configs"/"baseline_protocol_v1.json").exists(): return candidate
    raise FileNotFoundError("project root not found")
ROOT=find_project_root(Path.cwd())
SUMMARY_JSON=ROOT/"models"/"decoders"/STAGE2_SCHEMA/"fold_0"/"foundation_summary.json"

In [2]:
# Complete subject-level OOF comparison (2x2 factorial).
import hashlib
import matplotlib.pyplot as plt
from scipy.stats import wilcoxon

RUN_AGGREGATION = True
N_BOOTSTRAP = 10_000
ALPHA = 0.05
SEED = 42
MAIN_SEED = 42
VARIANCE_FOLD = 0
SWEEP_SEEDS = [123, 2024]
ARM_FACTORS = {
    "plain_unet_style":    {"activation": "relu",  "residual": False, "label": "U (ReLU, plain)"},
    "residual_vnet_style": {"activation": "prelu", "residual": True,  "label": "V (PReLU, residual)"},
    "residual_relu_style": {"activation": "relu",  "residual": True,  "label": "ReLU + residual"},
    "plain_prelu_style":   {"activation": "prelu", "residual": False, "label": "PReLU + plain"},
}
ARMS = list(ARM_FACTORS)
ARM_LABELS = {arm: ARM_FACTORS[arm]["label"] for arm in ARM_FACTORS}
DECODER_ROOT = ROOT / "models" / "decoders" / STAGE2_SCHEMA
REPORT_ROOT = ROOT / "reports" / "decoder_comparison" / STAGE2_SCHEMA
PRIMARY_METRICS = ["dice_macro", "assd_mm_macro"]

# Predefined contrasts (registered up front so nothing is chosen post-hoc), letting
# U=plain_unet_style, V=residual_vnet_style, R=residual_relu_style, P=plain_prelu_style.
CONTRASTS = {
    "decoder_family_V_minus_U": lambda a: a["residual_vnet_style"] - a["plain_unet_style"],
    "residual_main": lambda a: 0.5 * ((a["residual_relu_style"] - a["plain_unet_style"]) + (a["residual_vnet_style"] - a["plain_prelu_style"])),
    "activation_main": lambda a: 0.5 * ((a["plain_prelu_style"] - a["plain_unet_style"]) + (a["residual_vnet_style"] - a["residual_relu_style"])),
    "residual_activation_interaction": lambda a: (a["residual_vnet_style"] - a["residual_relu_style"]) - (a["plain_prelu_style"] - a["plain_unet_style"]),
}
CO_PRIMARY = ["residual_main", "activation_main"]  # tested x2 endpoints, Holm-corrected across the four
FACTOR_LABELS = {"residual_main": ("residual", "plain"), "activation_main": ("prelu", "relu")}


def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""): digest.update(chunk)
    return digest.hexdigest()


def canonical_sha256(payload):
    return hashlib.sha256(json.dumps(payload, sort_keys=True, separators=(",", ":")).encode()).hexdigest()


AMENDMENT_PATH = ROOT / "reports" / "agent_runs" / "stage2" / "s2_3_decoder_2x2_factorial" / "decoder_cv_protocol_amendment_v1.md"
COMBINED_GATE_PATH = ROOT / "models" / STAGE2_SCHEMA / "independent_stage2_gate_verdict.json"
CAPACITY_SUMMARY_PATH = DECODER_ROOT / "fold_0" / "foundation_summary.json"
REQUIRED_GATE_CONDITIONS = {"retain_fold4_in_primary", "run_fold4_excluded_sensitivity", "no_absolute_frontend_quality_claim", "use_revised_training_split_weighted_bce", "do_not_reuse_pre_amendment_unweighted_decoder_runs"}


def load_combined_gate():
    for required in (AMENDMENT_PATH, COMBINED_GATE_PATH, CAPACITY_SUMMARY_PATH):
        if not required.is_file(): raise FileNotFoundError(required)
    gate = json.loads(COMBINED_GATE_PATH.read_text(encoding="utf-8"))
    if gate.get("schema_version") != "stage2_combined_decoder_gate_v1" or gate.get("status") != "PASS" or gate.get("scope") != "full_decoder_cross_validation":
        raise RuntimeError("invalid combined Stage 2 decoder gate")
    if gate.get("protocol_amendment_sha256") != sha256_file(AMENDMENT_PATH) or gate.get("capacity_summary_sha256") != sha256_file(CAPACITY_SUMMARY_PATH):
        raise RuntimeError("combined gate is not bound to current amendment/capacity evidence")
    if not REQUIRED_GATE_CONDITIONS.issubset(set(gate.get("conditions", []))):
        raise RuntimeError("combined gate is missing required sensitivity/protocol conditions")
    return gate, sha256_file(COMBINED_GATE_PATH)


def run_identity(fold, arm, config, summary, paths, frame):
    frontend_path = ROOT / "models" / STAGE2_SCHEMA / f"fold_{fold}" / "shared_frontend.pth"
    if not frontend_path.is_file() or config.get("shared_frontend_sha256") != sha256_file(frontend_path):
        raise RuntimeError(f"run/front-end mismatch: fold={fold} arm={arm}")
    if len(frame) != summary.get("oof_rows") or frame.sample_id.nunique() != len(frame) or not frame.test_fold.eq(fold).all() or not frame.arm.eq(arm).all():
        raise RuntimeError(f"run OOF mismatch: fold={fold} arm={arm}")
    return {"fold": fold, "arm": arm, "seed": int(config["seed"]), "config_sha256": summary["config_sha256"], "run_summary_sha256": sha256_file(paths["summary"]), "checkpoint_sha256": sha256_file(paths["checkpoint"]), "oof_metrics_sha256": sha256_file(paths["metrics"]), "history_sha256": sha256_file(paths["history"]), "shared_frontend_sha256": sha256_file(frontend_path)}


def validate_run_provenance(fold, arm, config, identity, accepted, gate, gate_sha):
    qa = config.get("qa", {})
    evidence = qa.get("stage2_training_evidence")
    if evidence is not None:
        expected_evidence = {"capacity_summary_sha256": gate.get("capacity_summary_sha256"), "protocol_amendment_sha256": gate.get("protocol_amendment_sha256")}
        if any(evidence.get(key) != value for key, value in expected_evidence.items()):
            raise RuntimeError(f"revised run is bound to different Stage 2 evidence: fold={fold} arm={arm}")
        hyperparameters = config.get("hyperparameters", {})
        if config.get("stage") != "controlled_decoder_cv" or config.get("sweep") is not False:
            raise RuntimeError(f"invalid revised-run role: fold={fold} arm={arm}")
        if hyperparameters.get("loss") != "0.5_training_split_balanced_bce+0.5_soft_dice" or hyperparameters.get("bce_pos_weight_scope") != "fixed_fold_training_split_aggregate":
            raise RuntimeError(f"revised run does not use the registered weighted-loss policy: fold={fold} arm={arm}")
        capacity_gate = evidence.get("capacity_gate", {}); disposition = gate.get("capacity_disposition", {})
        expected_override = disposition.get("status") == "ACCEPT_WITH_PROJECT_OWNER_OVERRIDE"
        if capacity_gate.get("reported_pass") != disposition.get("reported_pass") or capacity_gate.get("override_applied") is not expected_override or capacity_gate.get("override_reason") != disposition.get("override_reason") or capacity_gate.get("failed_runs", []) != disposition.get("failed_runs", []):
            raise RuntimeError(f"revised run capacity disposition mismatch: fold={fold} arm={arm}")
        return "current_revised_protocol"
    run_gate = qa.get("combined_stage2_gate")
    if run_gate is not None:
        if run_gate.get("sha256") != gate_sha or run_gate.get("protocol_amendment_sha256") != sha256_file(AMENDMENT_PATH):
            raise RuntimeError(f"run is bound to a different combined gate: fold={fold} arm={arm}")
        return "combined_gate_bound"
    if identity not in accepted:
        raise RuntimeError(f"pre-amendment run was not independently accepted: fold={fold} arm={arm}")
    return "independently_accepted_pre_amendment"


def load_oof_results():
    gate, gate_sha = load_combined_gate(); accepted = gate.get("accepted_pre_amendment_runs", [])
    frames, summaries = [], []
    for fold in range(5):
        for arm in ARMS:
            run_dir = DECODER_ROOT / f"fold_{fold}" / arm
            paths = {"metrics": run_dir / "oof_metrics.csv", "summary": run_dir / "run_summary.json", "checkpoint": run_dir / "best_decoder.pth", "config": run_dir / "config.json", "history": run_dir / "history.csv"}
            for required in paths.values():
                if not required.is_file(): raise FileNotFoundError(required)
            config = json.loads(paths["config"].read_text(encoding="utf-8")); summary = json.loads(paths["summary"].read_text(encoding="utf-8")); frame = pd.read_csv(paths["metrics"])
            if summary.get("fold") != fold or summary.get("arm") != arm or not summary.get("success") or summary.get("checkpoint_sha256") != sha256_file(paths["checkpoint"]): raise RuntimeError(f"invalid run summary: fold={fold} arm={arm}")
            if summary.get("config_sha256") != canonical_sha256(config): raise RuntimeError(f"invalid run config hash: fold={fold} arm={arm}")
            identity = run_identity(fold, arm, config, summary, paths, frame)
            validate_run_provenance(fold, arm, config, identity, accepted, gate, gate_sha)
            frame["source_metrics_sha256"] = sha256_file(paths["metrics"]); frames.append(frame); summaries.append(summary)
    oof = pd.concat(frames, ignore_index=True); runs = pd.DataFrame(summaries)
    for arm in ARMS:
        arm_rows = oof[oof.arm.eq(arm)]
        if len(arm_rows) != 71 or arm_rows.sample_id.nunique() != 71: raise RuntimeError(f"{arm} does not contain exactly 71 unique OOF knees")
    pivot_fold = oof.pivot_table(index="sample_id", columns="arm", values="test_fold", aggfunc="first")
    pivot_front = oof.pivot_table(index="sample_id", columns="arm", values="shared_frontend_sha256", aggfunc="first")
    if len(pivot_fold) != 71 or pivot_fold.reindex(columns=ARMS).isna().any().any(): raise RuntimeError("OOF arms are not paired on all 71 knees")
    if not (pivot_fold.nunique(axis=1) == 1).all(): raise RuntimeError("paired knees used different folds across arms")
    if not (pivot_front.nunique(axis=1) == 1).all(): raise RuntimeError("paired knees used different shared front ends across arms")
    if oof[PRIMARY_METRICS].isna().any().any() or not np.isfinite(oof[PRIMARY_METRICS].to_numpy()).all(): raise RuntimeError("missing/non-finite co-primary OOF values")
    return oof, runs

def subject_level_table(oof, expected_subjects):
    """Average knees within subject first so bilateral knees do not receive extra weight."""
    metric_columns = [column for column in oof.columns if column.startswith(("dice_", "assd_mm_", "hd95_mm_", "false_bridge_", "empty_prediction_"))]
    subject = oof.groupby(["subject_id", "arm", "test_fold", "cohort"], as_index=False)[metric_columns].mean(numeric_only=True)
    if subject.subject_id.nunique() != expected_subjects: raise RuntimeError(f"expected {expected_subjects} subjects, found {subject.subject_id.nunique()}")
    counts = subject.groupby("arm").subject_id.nunique()
    if not (counts == expected_subjects).all(): raise RuntimeError(f"every arm must cover all expected subjects: {counts.to_dict()}")
    return subject


def arm_matrix(subject, metric):
    wide = subject.pivot_table(index=["subject_id", "test_fold", "cohort"], columns="arm", values=metric).reindex(columns=ARMS)
    if wide.isna().any().any(): raise RuntimeError(f"missing arm values for {metric}")
    return wide.reset_index()


def holm_adjust(p_values):
    order = np.argsort(p_values); adjusted = np.empty(len(p_values), dtype=float); running = 0.0
    for rank, index in enumerate(order):
        value = min(1.0, (len(p_values) - rank) * float(p_values[index])); running = max(running, value); adjusted[index] = running
    return adjusted


def fold_stratified_bootstrap(favourable_effect, folds, n=N_BOOTSTRAP, seed=SEED):
    """Resample subjects within each test fold; positive values already favour the added factor."""
    rng = np.random.default_rng(seed); favourable_effect = np.asarray(favourable_effect, dtype=float); folds = np.asarray(folds)
    sampled_sums = np.zeros(n, dtype=float); subject_count = 0
    for fold in np.unique(folds):
        effects = favourable_effect[folds == fold]
        indices = rng.integers(0, len(effects), size=(n, len(effects)))
        sampled_sums += effects[indices].sum(axis=1); subject_count += len(effects)
    values = sampled_sums / subject_count
    return {"mean_favourable_effect": float(values.mean()), "ci95_low": float(np.percentile(values, 2.5)), "ci95_high": float(np.percentile(values, 97.5)), "resamples": n}


def decide(results):
    interaction_significant = bool((results[results.contrast.eq("residual_activation_interaction")].p_raw < ALPHA).any())
    decisions = {"interaction_significant": interaction_significant}
    for contrast in CO_PRIMARY:
        sub = results[results.contrast.eq(contrast)]; dice = sub[sub.metric.eq("dice_macro")].iloc[0]; assd = sub[sub.metric.eq("assd_mm_macro")].iloc[0]
        both_significant = bool(dice.p_holm < ALPHA and assd.p_holm < ALPHA)
        same_direction = bool(np.sign(dice.favourable_effect_median) == np.sign(assd.favourable_effect_median) and dice.favourable_effect_median != 0)
        added, baseline = FACTOR_LABELS[contrast]
        if both_significant and same_direction and not interaction_significant:
            decisions[contrast] = f"prefers_{added}" if dice.favourable_effect_median > 0 else f"prefers_{baseline}"
        else:
            decisions[contrast] = "INCONCLUSIVE"
    return decisions


def primary_analysis(subject):
    matrices = {metric: arm_matrix(subject, metric) for metric in PRIMARY_METRICS}
    rows = []
    for metric in PRIMARY_METRICS:
        matrix = matrices[metric]; favour_sign = 1.0 if metric == "dice_macro" else -1.0; folds = matrix["test_fold"].to_numpy()
        arm_values = {arm: matrix[arm].to_numpy(float) for arm in ARMS}
        for name, function in CONTRASTS.items():
            effect = function(arm_values); favourable = effect * favour_sign
            test = None if np.allclose(effect, 0.0) else wilcoxon(effect, zero_method="pratt", alternative="two-sided", method="auto")
            rows.append({"metric": metric, "contrast": name, "co_primary": name in CO_PRIMARY, "raw_effect_mean": float(effect.mean()), "favourable_effect_mean": float(favourable.mean()), "favourable_effect_median": float(np.median(favourable)), "wilcoxon_statistic": 0.0 if test is None else float(test.statistic), "p_raw": 1.0 if test is None else float(test.pvalue), **fold_stratified_bootstrap(favourable, folds)})
    results = pd.DataFrame(rows); results["p_holm"] = np.nan
    co_primary = results[results.co_primary]
    results.loc[co_primary.index, "p_holm"] = holm_adjust(co_primary.p_raw.to_numpy())
    return results, decide(results)


def pathology_interaction(subject):
    """Descriptive only (n_fractured = 13, unpaired across cohorts): does an effect differ by pathology?"""
    rows = []
    for metric in PRIMARY_METRICS:
        favour_sign = 1.0 if metric == "dice_macro" else -1.0
        for name in ("residual_main", "decoder_family_V_minus_U"):
            per_cohort = {}
            for cohort in ("healthy", "fractured"):
                matrix = subject[subject.cohort.eq(cohort)].pivot_table(index="subject_id", columns="arm", values=metric).reindex(columns=ARMS)
                if len(matrix) == 0 or matrix.isna().any().any():
                    per_cohort[cohort] = float("nan"); continue
                arm_values = {arm: matrix[arm].to_numpy(float) for arm in ARMS}
                per_cohort[cohort] = float(np.median(CONTRASTS[name](arm_values) * favour_sign))
            rows.append({"metric": metric, "contrast": name, "healthy_median_favourable": per_cohort["healthy"], "fractured_median_favourable": per_cohort["fractured"], "cohort_difference": per_cohort["fractured"] - per_cohort["healthy"], "note": "descriptive_only_n_fractured_13"})
    return pd.DataFrame(rows)


def secondary_tables(oof, subject):
    bone_rows = []
    for bone in BONES:
        for arm in ARMS:
            values = subject[subject.arm.eq(arm)]; bone_rows.append({"bone": bone, "arm": arm, "arm_label": ARM_LABELS[arm], "subjects": len(values), "dice_mean": float(values[f"dice_{bone}"].mean()), "assd_mm_mean": float(values[f"assd_mm_{bone}"].mean())})
    cohort = subject.groupby(["cohort", "arm"], as_index=False)[PRIMARY_METRICS].agg(["count", "mean", "median"]).reset_index()
    topology_columns = [column for column in subject.columns if column.startswith(("false_bridge_", "empty_prediction_"))]
    topology = subject.groupby("arm", as_index=False)[topology_columns].mean(numeric_only=True)
    return pd.DataFrame(bone_rows), cohort, topology


def load_variance_band():
    """Fold-0 across-seed dispersion of subject-macro metrics, so an effect can be judged against
    training-run variability (seed 42 from main CV plus the sweep seeds)."""
    sweep_root = DECODER_ROOT / "seed_sweep"
    rows = []
    for arm in ARMS:
        per_seed = {}
        main_metrics = DECODER_ROOT / f"fold_{VARIANCE_FOLD}" / arm / "oof_metrics.csv"
        if main_metrics.is_file():
            per_seed[MAIN_SEED] = pd.read_csv(main_metrics)
        for seed in SWEEP_SEEDS:
            sweep_metrics = sweep_root / f"fold_{VARIANCE_FOLD}" / arm / f"seed_{seed}" / "oof_metrics.csv"
            if sweep_metrics.is_file():
                per_seed[seed] = pd.read_csv(sweep_metrics)
        for seed, frame in per_seed.items():
            subject_means = frame.groupby("subject_id")[PRIMARY_METRICS].mean(numeric_only=True)
            rows.append({"arm": arm, "seed": seed, "n_subjects": int(len(subject_means)), **{f"{metric}_mean": float(subject_means[metric].mean()) for metric in PRIMARY_METRICS}})
    band = pd.DataFrame(rows)
    if band.empty:
        return {"status": "NOT_PROVIDED"}, band
    dispersion = band.groupby("arm")[[f"{metric}_mean" for metric in PRIMARY_METRICS]].agg(["mean", "std", "count"])
    dispersion.columns = ["_".join(column) for column in dispersion.columns]
    return {"status": "AVAILABLE", "seeds": sorted(int(seed) for seed in band.seed.unique()), "arms": len(band.arm.unique())}, band.merge(dispersion.reset_index(), on="arm")


def resource_table(runs):
    rows = []
    for record in runs.to_dict("records"):
        resource = record.get("resource_usage", {}); architecture = record.get("architecture", {})
        rows.append({"fold": record["fold"], "arm": record["arm"], "arm_label": record.get("arm_label", ARM_LABELS[record["arm"]]), "seed": record.get("seed"), "logit_resolution": record.get("logit_resolution"), "parameters": architecture.get("parameters"), "peak_gpu_bytes": resource.get("peak_gpu_bytes"), "gpu_headroom_fraction": resource.get("gpu_headroom_fraction"), "wall_seconds": resource.get("wall_seconds")})
    return pd.DataFrame(rows)


def save_figures(bone, subject):
    REPORT_ROOT.mkdir(parents=True, exist_ok=True)
    labels = [ARM_LABELS[arm] for arm in ARMS]
    figure, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    for axis, metric, title in zip(axes, PRIMARY_METRICS, ["Subject macro Dice", "Subject macro ASSD (mm)"]):
        data = [subject[subject.arm.eq(arm)][metric].to_numpy() for arm in ARMS]; axis.boxplot(data, labels=labels, showmeans=True); axis.set_title(title); axis.grid(alpha=0.25); axis.tick_params(axis="x", rotation=20)
    figure.tight_layout(); figure.savefig(REPORT_ROOT / "co_primary_boxplots.png", dpi=180); plt.close(figure)
    pivot_dice = bone.pivot(index="bone", columns="arm", values="dice_mean").loc[BONES]; pivot_assd = bone.pivot(index="bone", columns="arm", values="assd_mm_mean").loc[BONES]
    figure, axes = plt.subplots(1, 2, figsize=(13, 4.5)); pivot_dice.plot(kind="bar", ax=axes[0]); pivot_assd.plot(kind="bar", ax=axes[1]); axes[0].set_title("Per-bone Dice"); axes[1].set_title("Per-bone ASSD (mm)")
    for axis in axes: axis.grid(axis="y", alpha=0.25); axis.tick_params(axis="x", rotation=0)
    figure.tight_layout(); figure.savefig(REPORT_ROOT / "per_bone_comparison.png", dpi=180); plt.close(figure)


def validate_regen_illustrations():
    path = REPORT_ROOT / "regen_illustrative_manifest.csv"
    if not path.is_file(): return {"status": "NOT_PROVIDED", "used_for_winner": False}
    frame = pd.read_csv(path); forbidden = {"patient_name", "patient_id", "filename", "folder_name"}
    if forbidden & set(column.casefold() for column in frame.columns): raise RuntimeError("Regen illustration manifest contains a forbidden PII-bearing column")
    if "study_id" not in frame.columns or frame.study_id.astype(str).str.strip().eq("").any(): raise RuntimeError("Regen illustration manifest requires non-empty study_id")
    return {"status": "ILLUSTRATIVE_ONLY", "rows": len(frame), "used_for_winner": False, "sha256": sha256_file(path)}


def compare_primary_sensitivity(primary_decisions, sensitivity_decisions):
    rows, overall = {}, "ROBUST"
    for contrast in CO_PRIMARY:
        primary = primary_decisions[contrast]; sensitivity = sensitivity_decisions[contrast]
        if primary == sensitivity:
            status = "ROBUST" if primary != "INCONCLUSIVE" else "CONSISTENTLY_INCONCLUSIVE"
        elif primary != "INCONCLUSIVE" and sensitivity == "INCONCLUSIVE":
            status = "FOLD4_DEPENDENT"
        else:
            status = "INCONCLUSIVE"
        rows[contrast] = {"primary": primary, "sensitivity": sensitivity, "robustness": status}
        if status == "INCONCLUSIVE": overall = "INCONCLUSIVE"
        elif status == "FOLD4_DEPENDENT" and overall != "INCONCLUSIVE": overall = "FOLD4_DEPENDENT"
    return {"overall": overall, "factors": rows}


def run_comparison():
    REPORT_ROOT.mkdir(parents=True, exist_ok=True)
    oof, runs = load_oof_results()
    subject = subject_level_table(oof, expected_subjects=43)
    primary, decisions = primary_analysis(subject)

    sensitivity_oof = oof[oof.test_fold.ne(4)].copy()
    sensitivity_knees = sensitivity_oof.sample_id.nunique()
    sensitivity_subjects = sensitivity_oof.subject_id.nunique()
    if sensitivity_knees != 57 or sensitivity_subjects != 34:
        raise RuntimeError(f"unexpected fold-4-excluded cohort: knees={sensitivity_knees}, subjects={sensitivity_subjects}")
    sensitivity_subject = subject_level_table(sensitivity_oof, expected_subjects=34)
    sensitivity_primary, sensitivity_decisions = primary_analysis(sensitivity_subject)
    robustness = compare_primary_sensitivity(decisions, sensitivity_decisions)

    bone, cohort, topology = secondary_tables(oof, subject)
    pathology = pathology_interaction(subject)
    resources = resource_table(runs)
    variance_status, variance = load_variance_band()
    regen = validate_regen_illustrations()

    oof.to_csv(REPORT_ROOT / "oof_all_knees.csv", index=False)
    subject.to_csv(REPORT_ROOT / "subject_level_metrics.csv", index=False)
    primary.to_csv(REPORT_ROOT / "co_primary_results.csv", index=False)
    sensitivity_oof.to_csv(REPORT_ROOT / "sensitivity_fold4_excluded_oof.csv", index=False)
    sensitivity_subject.to_csv(REPORT_ROOT / "sensitivity_fold4_excluded_subject_metrics.csv", index=False)
    sensitivity_primary.to_csv(REPORT_ROOT / "sensitivity_fold4_excluded_co_primary_results.csv", index=False)
    bone.to_csv(REPORT_ROOT / "per_bone_results.csv", index=False)
    cohort.to_csv(REPORT_ROOT / "cohort_results.csv", index=False)
    topology.to_csv(REPORT_ROOT / "topology_results.csv", index=False)
    pathology.to_csv(REPORT_ROOT / "pathology_interaction_results.csv", index=False)
    resources.to_csv(REPORT_ROOT / "resource_results.csv", index=False)
    if not variance.empty: variance.to_csv(REPORT_ROOT / "variance_band_results.csv", index=False)
    save_figures(bone, subject)

    summary = {
        "schema_version": STAGE2_SCHEMA,
        "analysis": "controlled_decoder_factorial_oof_comparison",
        "seed": SEED,
        "arms": ARMS,
        "primary": {"knees": int(oof.sample_id.nunique()), "subjects": int(subject.subject_id.nunique()), "decisions": decisions, "contrasts": primary.to_dict("records")},
        "sensitivity_fold4_excluded": {"selection_rule": "test_fold != 4", "cannot_select_winner": True, "knees": int(sensitivity_knees), "subjects": int(sensitivity_subjects), "decisions": sensitivity_decisions, "contrasts": sensitivity_primary.to_dict("records")},
        "robustness_interpretation": robustness,
        "co_primary_metrics": PRIMARY_METRICS,
        "co_primary_contrasts": CO_PRIMARY,
        "decision_rule": "both Holm-corrected co-primary p-values < 0.05 in the same direction on both endpoints, with a non-significant residual x activation interaction",
        "pathology_interaction": pathology.to_dict("records"),
        "variance_band": variance_status,
        "fractured_subjects": int(subject[subject.cohort.eq("fractured")].subject_id.nunique()),
        "healthy_subjects": int(subject[subject.cohort.eq("healthy")].subject_id.nunique()),
        "regen": regen,
        "limitations": ["one training seed per fold (fold-0 variance band only)", "fractured subgroup is descriptive", "fixed-front-end conditional inference", "fold 4 has pre-declared front-end collapse evidence and is tested by mandatory sensitivity analysis", "the 2x2 estimates residual and activation main effects but power is limited", "logits computed at 128^3 then upsampled", "no clinical-utility claim"],
        "success": len(oof) == 71 * len(ARMS) and subject.subject_id.nunique() == 43 and sensitivity_knees == 57 and sensitivity_subjects == 34 and not primary[primary.co_primary].p_holm.isna().any() and not sensitivity_primary[sensitivity_primary.co_primary].p_holm.isna().any(),
    }
    (REPORT_ROOT / "comparison_summary.json").write_text(json.dumps(summary, indent=2, sort_keys=True) + "\n", encoding="utf-8")
    if not summary["success"]: raise RuntimeError("comparison success criteria failed")
    return summary, primary, sensitivity_primary, bone, cohort, topology, pathology, resources, variance


In [3]:
if RUN_AGGREGATION:
    summary, primary, sensitivity, bone, cohort, topology, pathology, resources, variance = run_comparison()
    display(primary); display(sensitivity); display(bone); display(cohort); display(topology); display(pathology); display(resources)
    if not variance.empty: display(variance)
    print("primary decisions:", summary["primary"]["decisions"])
    print("fold-4-excluded decisions:", summary["sensitivity_fold4_excluded"]["decisions"])
    print("robustness:", summary["robustness_interpretation"])
else:
    print("Definitions loaded. Set RUN_AGGREGATION=True only after all twenty run summaries report success or are independently accepted pre-amendment runs.")


/tmp/ipykernel_47768/698845257.py:262: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  data = [subject[subject.arm.eq(arm)][metric].to_numpy() for arm in ARMS]; axis.boxplot(data, labels=labels, showmeans=True); axis.set_title(title); axis.grid(alpha=0.25); axis.tick_params(axis="x", rotation=20)
/tmp/ipykernel_47768/698845257.py:262: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  data = [subject[subject.arm.eq(arm)][metric].to_numpy() for arm in ARMS]; axis.boxplot(data, labels=labels, showmeans=True); axis.set_title(title); axis.grid(alpha=0.25); axis.tick_params(axis="x", rotation=20)


,metric,contrast,co_primary,raw_effect_mean,favourable_effect_mean,favourable_effect_median,wilcoxon_statistic,p_raw,mean_favourable_effect,ci95_low,ci95_high,resamples,p_holm
0,dice_macro,decoder_family_V_minus_U,False,0.039135,0.039135,0.045578,39.0,1.724402e-09,0.039125,0.030686,0.047328,10000,NaN
1,dice_macro,residual_main,True,0.043999,0.043999,0.048942,31.0,5.400125e-10,0.043978,0.034962,0.052766,10000,2.160050e-09
2,dice_macro,activation_main,True,-0.004864,-0.004864,-0.005631,202.0,7.588058e-04,-0.004853,-0.007716,-0.001548,10000,1.517612e-03
3,dice_macro,residual_activation_interaction,False,-0.009178,-0.009178,-0.011762,164.0,9.687860e-05,-0.009204,-0.014851,-0.002732,10000,NaN
4,assd_mm_macro,decoder_family_V_minus_U,False,-2.202982,2.202982,2.974588,43.0,2.954266e-09,2.194312,0.161397,3.390099,10000,NaN
5,assd_mm_macro,residual_main,True,-2.261483,2.261483,3.194654,43.0,2.954266e-09,2.252300,0.199471,3.455072,10000,8.862798e-09
6,assd_mm_macro,activation_main,True,0.058502,-0.058502,-0.164275,379.0,2.618016e-01,-0.057988,-0.150920,0.038538,10000,2.618016e-01
7,assd_mm_macro,residual_activation_interaction,False,0.350865,-0.350865,-0.304504,293.0,2.918418e-02,-0.351342,-0.569532,-0.100205,10000,NaN


,metric,contrast,co_primary,raw_effect_mean,favourable_effect_mean,favourable_effect_median,wilcoxon_statistic,p_raw,mean_favourable_effect,ci95_low,ci95_high,resamples,p_holm
0,dice_macro,decoder_family_V_minus_U,False,0.036246,0.036246,0.044630,36.0,5.791662e-07,0.036240,0.025969,0.046122,10000,NaN
1,dice_macro,residual_main,True,0.043412,0.043412,0.048557,28.0,1.726439e-07,0.043387,0.032603,0.053844,10000,6.905757e-07
2,dice_macro,activation_main,True,-0.007166,-0.007166,-0.008073,65.0,1.910841e-05,-0.007147,-0.010682,-0.003130,10000,3.821682e-05
3,dice_macro,residual_activation_interaction,False,-0.007427,-0.007427,-0.007009,135.0,4.590919e-03,-0.007459,-0.014519,0.000641,10000,NaN
4,assd_mm_macro,decoder_family_V_minus_U,False,-1.712481,1.712481,2.744111,34.0,4.336471e-07,1.702253,-0.848452,3.191349,10000,NaN
5,assd_mm_macro,residual_main,True,-1.973014,1.973014,3.158650,34.0,4.336471e-07,1.962216,-0.630875,3.472997,10000,1.300941e-06
6,assd_mm_macro,activation_main,True,0.260532,-0.260532,-0.279207,81.0,8.892361e-05,-0.259963,-0.362923,-0.149524,10000,8.892361e-05
7,assd_mm_macro,residual_activation_interaction,False,-0.034095,0.034095,-0.085904,273.0,6.853144e-01,0.033076,-0.226142,0.336193,10000,NaN


,bone,arm,arm_label,subjects,dice_mean,assd_mm_mean
0,femur,plain_unet_style,"U (ReLU, plain)",43,0.596284,9.145565
1,femur,residual_vnet_style,"V (PReLU, residual)",43,0.619640,8.209327
2,femur,residual_relu_style,ReLU + residual,43,0.630503,7.933633
3,femur,plain_prelu_style,PReLU + plain,43,0.592793,9.257895
4,tibia,plain_unet_style,"U (ReLU, plain)",43,0.595383,8.804695
5,tibia,residual_vnet_style,"V (PReLU, residual)",43,0.616334,7.939266
6,tibia,residual_relu_style,ReLU + residual,43,0.627043,7.714389
7,tibia,plain_prelu_style,PReLU + plain,43,0.593026,8.804062
8,patella,plain_unet_style,"U (ReLU, plain)",43,0.328137,11.639086
9,patella,residual_vnet_style,"V (PReLU, residual)",43,0.367496,12.725141


index     cohort                  arm dice_macro                      \
                                             count      mean    median   
0     0  fractured    plain_prelu_style         13  0.329737  0.338686   
1     1  fractured     plain_unet_style         13  0.336279  0.332850   
2     2  fractured  residual_relu_style         13  0.377243  0.405404   
3     3  fractured  residual_vnet_style         13  0.365457  0.390433   
4     4    healthy    plain_prelu_style         30  0.439145  0.438967   
5     5    healthy     plain_unet_style         30  0.436704  0.441789   
6     6    healthy  residual_relu_style         30  0.488596  0.490984   
7     7    healthy  residual_vnet_style         30  0.480154  0.482678   

  assd_mm_macro                        
          count       mean     median  
0            13  15.721072  15.231740  
1            13  15.465615  15.278260  
2            13  12.193900  12.277574  
3            13  12.304298  12.428087  
4            30  12.914144  12.220634  
5            30  13.192443  12.126388  
6            30  11.117273   8.623812  
7            30  11.404740   9.154662

,arm,empty_prediction_femur,false_bridge_femur,empty_prediction_tibia,false_bridge_tibia,empty_prediction_patella,false_bridge_patella,empty_prediction_fibula,false_bridge_fibula
0,plain_prelu_style,0.0,0.174419,0.0,0.209302,0.000000,0.069767,0.0,0.162791
1,plain_unet_style,0.0,0.174419,0.0,0.209302,0.000000,0.069767,0.0,0.162791
2,residual_relu_style,0.0,0.174419,0.0,0.209302,0.011628,0.069767,0.0,0.139535
3,residual_vnet_style,0.0,0.174419,0.0,0.209302,0.011628,0.069767,0.0,0.162791


,metric,contrast,healthy_median_favourable,fractured_median_favourable,cohort_difference,note
0,dice_macro,residual_main,0.051698,0.041733,-0.009965,descriptive_only_n_fractured_13
1,dice_macro,decoder_family_V_minus_U,0.049469,0.036155,-0.013315,descriptive_only_n_fractured_13
2,assd_mm_macro,residual_main,3.340893,3.142377,-0.198515,descriptive_only_n_fractured_13
3,assd_mm_macro,decoder_family_V_minus_U,3.003403,2.552496,-0.450906,descriptive_only_n_fractured_13


,fold,arm,arm_label,seed,logit_resolution,parameters,peak_gpu_bytes,gpu_headroom_fraction,wall_seconds
0,0,plain_unet_style,"U (ReLU, plain)",42,128,8429956,8468765696,0.642226,2242.5
1,0,residual_vnet_style,"V (PReLU, residual)",42,128,8605476,8502994432,0.640780,2234.2
2,0,residual_relu_style,ReLU + residual,42,128,8604516,8436281856,0.643598,2226.8
3,0,plain_prelu_style,PReLU + plain,42,128,8430916,8468094976,0.642254,2166.7
4,1,plain_unet_style,"U (ReLU, plain)",42,128,8429956,8398040064,0.645213,2303.9
5,1,residual_vnet_style,"V (PReLU, residual)",42,128,8605476,8433841664,0.643701,2340.0
6,1,residual_relu_style,ReLU + residual,42,128,8604516,8400525824,0.645108,2366.9
7,1,plain_prelu_style,PReLU + plain,42,128,8430916,8429914112,0.643867,2310.5
8,2,plain_unet_style,"U (ReLU, plain)",42,128,8429956,8396205056,0.645291,2299.0
9,2,residual_vnet_style,"V (PReLU, residual)",42,128,8605476,8433841664,0.643701,2315.9


,arm,seed,n_subjects,dice_macro_mean,assd_mm_macro_mean,dice_macro_mean_mean,dice_macro_mean_std,dice_macro_mean_count,assd_mm_macro_mean_mean,assd_mm_macro_mean_std,assd_mm_macro_mean_count
0,plain_unet_style,42,9,0.450092,11.387169,0.450092,NaN,1,11.387169,NaN,1
1,residual_vnet_style,42,9,0.481315,8.823584,0.481315,NaN,1,8.823584,NaN,1
2,residual_relu_style,42,9,0.495089,8.509803,0.495089,NaN,1,8.509803,NaN,1
3,plain_prelu_style,42,9,0.441305,12.112789,0.441305,NaN,1,12.112789,NaN,1


primary decisions: {'interaction_significant': True, 'residual_main': 'INCONCLUSIVE', 'activation_main': 'INCONCLUSIVE'}
fold-4-excluded decisions: {'interaction_significant': True, 'residual_main': 'INCONCLUSIVE', 'activation_main': 'INCONCLUSIVE'}
robustness: {'overall': 'ROBUST', 'factors': {'residual_main': {'primary': 'INCONCLUSIVE', 'sensitivity': 'INCONCLUSIVE', 'robustness': 'CONSISTENTLY_INCONCLUSIVE'}, 'activation_main': {'primary': 'INCONCLUSIVE', 'sensitivity': 'INCONCLUSIVE', 'robustness': 'CONSISTENTLY_INCONCLUSIVE'}}}
